# Confounded dSprites — Results

The design where the context sets the sprite's shape and scale as well as the action, so
$x \to a$, $x \to y$ and $a \to y$ and recovering $\nu(\pi)$ genuinely requires adjusting for $x$.
Distinct from the single-sprite design of arXiv:2506.02793, whose outcome depends on the action
alone; results for that one are in `dSprites_Results_Analysis.ipynb` and the two must never be
pooled.

Headline configuration: $n = 20000$, $M = 2000$, `--init observed`, DR target, projected Nyström,
no weight clipping, no pixel noise.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import glob
import pandas as pd
import analysis_tables as at

RESULTS = ["../dSprites/dr_mmd_flow/results_confounded",
           "../dSprites/dr_mmd_flow/results_confounded_prefidfix"]
ALL = pd.concat([pd.read_csv(f) for d in RESULTS for f in sorted(glob.glob(d + "/*.csv"))],
                ignore_index=True)
ALL = ALL[ALL.design == "confounded"]

# The headline cell. `repeat` is absent from these runs, so give `summarise` the column it
# groups on rather than special-casing it downstream.
if "repeat" not in ALL.columns:
    ALL["repeat"] = 0
DEFAULT = ALL[(ALL.n_obs == 20000) & (ALL.init == "observed")
              & (ALL.n_particles == 2000) & (ALL.estimator == "dr")]
LADDER = ALL[(ALL.init == "observed") & (ALL.n_particles == 2000) & (ALL.estimator == "dr")]

# Per-row, so `summarise` reports an honest spread. A ratio of two means has no sd.
ALL["sw2_excess"] = ALL.sw2_flow_mean - ALL.sw2_floor_noisy_matched_mean
ALL["sw2_ratio"] = ALL.sw2_flow_mean / ALL.sw2_floor_noisy_matched_mean
DEFAULT = ALL[(ALL.n_obs == 20000) & (ALL.init == "observed")
              & (ALL.n_particles == 2000) & (ALL.estimator == "dr")]
LADDER = ALL[(ALL.init == "observed") & (ALL.n_particles == 2000) & (ALL.estimator == "dr")]

print(f"{len(ALL)} rows   headline cell: {len(DEFAULT)} seeds")
print(f"sizes {sorted(ALL.n_obs.unique())}   particles {sorted(ALL.n_particles.unique())}   "
      f"inits {sorted(ALL.init.unique())}")

52 rows   headline cell: 12 seeds
sizes [np.int64(5000), np.int64(10000), np.int64(15000), np.int64(20000)]   particles [np.int64(250), np.int64(500), np.int64(1000), np.int64(2000)]   inits ['moment', 'normal', 'observed']


## 1. Headline

All three target embeddings, with every metric. They come from **one** kernel-ridge solve:
`components=True` returns $(v_{\mathrm{PI}}, v_{\mathrm{corr}})$, so plug-in is $v_{\mathrm{PI}}$,
DR is $v_{\mathrm{PI}} + v_{\mathrm{corr}}$, and IPW is $\hat w / n$ with no conditional mean
embedding at all. The three rows are **paired within each run** — same data, same nuisances, same
initial particles, same reference draws — so the only thing that varies is the weight vector handed
to the flow.

The floor is **count-matched**: an i.i.d. draw of $M$ points, the right yardstick for $M$
particles. `logged through the AE` is the logged sample passed through the autoencoder — *not* an
oracle draw — so it isolates the decoder's cost: it sits on the logged row, which says the decoder
is nearly free and the flow's gain is transport rather than smoothing.

Two things to carry into the text. The three estimators **do not separate** here: they differ by
less than the seed spread on every metric, because once each reaches the sampling-noise floor the
metric cannot resolve them. And every flow row sits *below* the count-matched floor, which
Section 6 shows is a finite-$M$ artefact — the claim is that the sampler **reaches** the floor,
not that it beats an i.i.d. draw.

In [2]:
# Paired runs only, so the three estimators are compared on identical draws.
PAIRED = DEFAULT_ALL = ALL[(ALL.n_obs == 20000) & (ALL.n_particles == 2000)
                           & (ALL.init == "observed")]
PAIRED = PAIRED[PAIRED.groupby("data_seed").estimator.transform("nunique") == 3]

def metric_row(g):
    fx = g[g.n_fid_ref == 2000]
    return [at.summarise(g, "sw2_flow_mean"), at.summarise(g, "sw2_ratio"),
            at.summarise(g, "latent_sw2_flow_mean"),
            at.summarise(fx, "fid_legacy_flow"), at.summarise(fx, "fid_clean_flow")]

NAMES = [("DR-IMMD", "dr"), ("Plug-in IMMD", "plugin"), ("IPW-IMMD", "ipw")]
rows = []
base = PAIRED[PAIRED.estimator == "dr"]          # the shared rows live on any estimator's row
bf = base[base.n_fid_ref == 2000]
rows.append(("logged (no flow)",
             [at.summarise(base, "sw2_logged_mean"),
              (at.summarise(base, "sw2_logged_mean")[0]
               / at.summarise(base, "sw2_floor_noisy_matched_mean")[0], 0.0),
              at.summarise(base, "latent_sw2_initial_mean"),
              at.summarise(bf, "fid_legacy_initial"), at.summarise(bf, "fid_clean_initial")]))
for label, e in NAMES:
    g = PAIRED[PAIRED.estimator == e]
    rows.append((f"{label}  ({g.data_seed.nunique()} seeds)", metric_row(g)))
rows.append(("logged through the AE",
             [at.summarise(base, "sw2_roundtrip_mean"),
              (at.summarise(base, "sw2_roundtrip_mean")[0]
               / at.summarise(base, "sw2_floor_noisy_matched_mean")[0], 0.0),
              (float("nan"), 0.0),
              at.summarise(bf, "fid_legacy_roundtrip"), at.summarise(bf, "fid_clean_roundtrip")]))
rows.append(("floor (count-matched)",
             [at.summarise(base, "sw2_floor_noisy_matched_mean"), (1.0, 0.0),
              at.summarise(base, "latent_sw2_floor_matched_mean"),
              at.summarise(bf, "fid_legacy_floor"), at.summarise(bf, "fid_clean_floor")]))
print(at.plain_table(rows, ["pixel SW2", "ratio to floor", "latent SW2",
                            "FID legacy", "FID clean"],
                     title="Confounded dSprites, n = 20000, M = 2000 (estimators paired per run)",
                     note="DR over all 12 available seeds gives "
                          f"{at.summarise(DEFAULT, 'sw2_flow_mean')[0]:.5f} +- "
                          f"{at.summarise(DEFAULT, 'sw2_flow_mean')[1]:.5f}, i.e. the same value",
                     label_width=24))

Confounded dSprites, n = 20000, M = 2000 (estimators paired per run)

Method                  pixel SW2 ↓          ratio to floor ↓     latent SW2 ↓         FID legacy ↓         FID clean ↓
-----------------------------------------------------------------------------------------------------------------------
logged (no flow)        0.02765 ± 0.00040    2.851 ± 0.000        0.14596 ± 0.00787    36.776 ± 1.524       64.209 ± 5.466
DR-IMMD  (3 seeds)      0.00841 ± 0.00008    0.86722 ± 0.02054    0.03906 ± 0.00047    36.488 ± 1.648       61.998 ± 5.424
Plug-in IMMD  (3 seeds) 0.00842 ± 0.00003    0.86800 ± 0.01433    0.03900 ± 0.00043    36.409 ± 1.770       62.402 ± 5.686
IPW-IMMD  (3 seeds)     0.00847 ± 0.00009    0.87373 ± 0.02089    0.03952 ± 0.00056    36.391 ± 1.774       62.103 ± 5.818
logged through the AE   0.02793 ± 0.00042    2.879 ± 0.000        --                   36.448 ± 1.995       62.626 ± 6.656
floor (count-matched)   0.00970 ± 0.00014    1.000 ± 0.000        0.05123 ±

## 2. Latent SW2

In the standardized 32-d latent the flow actually operates in. Not comparable across designs as a
raw number — SW2 averages over coordinates, so the same per-coordinate error costs differently at
different $d$. **The quantity that travels is the ratio to this design's own count-matched floor.**

In [3]:
LAT = [("initial particles", "initial"), ("IMMD-Flow", "flow"),
       ("floor (count-matched)", "floor_matched")]
rows = [(label, [at.summarise(DEFAULT, f"latent_sw2_{k}_mean")]) for label, k in LAT]
f = at.summarise(DEFAULT, "latent_sw2_floor_matched_mean")[0]
print(at.plain_table(rows, ["latent SW2"], title="Latent SW2, n = 20000",
                     note=f"IMMD-Flow / matched floor = "
                          f"{at.summarise(DEFAULT,'latent_sw2_flow_mean')[0] / f:.2f}"))

Latent SW2, n = 20000

Method                latent SW2 ↓
----------------------------------
initial particles     0.14596 ± 0.00787
IMMD-Flow             0.03906 ± 0.00047
floor (count-matched) 0.05123 ± 0.00301

IMMD-Flow / matched floor = 0.76


## 3. FID

Single channel replicated to RGB. **Restricted to rows scored against a FIXED 2000-image
reference** (`n_fid_ref == 2000`). Earlier rows used the whole `n_causal` draw as the reference —
i.e. the sweep axis itself — which made the column move with $n$ for a purely statistical reason:
the floor between two draws of the *same* law drifted 0.53 → 0.14 across the ladder. It is flat
once the count is pinned.

**What this table does and does not say.** The floor — two independent oracle draws — is ~1.3,
and every decoded row sits near 36. So FID here is almost entirely the autoencoder. Crucially the
flow, the initial particles and the logged round-trip are all within noise of *each other*: FID
does not separate the flow from the logged law at all. It certifies that the samples are valid
images; it cannot see the intervention, which is what pixel and latent SW2 measure.

**There is no perfect-sampler row in this simulation.** `roundtrip` is the *logged* sample
decoded, not an oracle draw decoded, so it is not a ceiling for a correct sampler. Measuring that
ceiling needs an oracle draw through the autoencoder, which the paper-figures notebook computes
and this one does not.

In [4]:
FIXED = DEFAULT[DEFAULT.n_fid_ref == 2000]
FID = [("initial particles", "initial"), ("IMMD-Flow", "flow"),
       ("logged through the AE", "roundtrip"), ("oracle vs oracle (floor)", "floor")]
rows = [(label, [at.summarise(FIXED, f"fid_{impl}_{k}") for impl in ("legacy", "clean")])
        for label, k in FID]
gap = (at.summarise(FIXED, "fid_legacy_flow")[0] - at.summarise(FIXED, "fid_legacy_initial")[0])
print(at.plain_table(rows, ["FID legacy", "FID clean"],
                     title=f"FID, n = 20000, 2000 generated vs 2000 references, "
                           f"{len(FIXED)} seeds",
                     note=f"flow - initial = {gap:+.2f} (legacy): FID does not separate the "
                          "flow from its own starting cloud. It certifies validity, not the "
                          "intervention; the floor at ~1.3 shows how much of it is the decoder"))

FID, n = 20000, 2000 generated vs 2000 references, 9 seeds

Method                FID legacy ↓         FID clean ↓
------------------------------------------------------
initial particles     36.776 ± 1.524       64.209 ± 5.466
IMMD-Flow             36.488 ± 1.648       61.998 ± 5.424
logged through the AE 36.448 ± 1.995       62.626 ± 6.656
oracle vs oracle (floor)1.305 ± 0.055        1.571 ± 0.060

flow - initial = -0.29 (legacy): FID does not separate the flow from its own starting cloud. It certifies validity, not the intervention; the floor at ~1.3 shows how much of it is the decoder


## 4. Observation count

$M$ and every other setting held fixed. All three metrics at once.

In [5]:
def ladder(frame, by, order, title, note=""):
    rows = []
    for v in order:
        g = frame[frame[by] == v]
        if not len(g): continue
        fx = g[g.n_fid_ref == 2000] if "n_fid_ref" in g else g
        rows.append((f"{v}  ({g.data_seed.nunique()} seeds)",
                     [at.summarise(g, "sw2_flow_mean"),
                      at.summarise(g, "sw2_floor_noisy_matched_mean"),
                      at.summarise(g, "latent_sw2_flow_mean"),
                      at.summarise(fx, "fid_legacy_flow") if len(fx) else (float("nan"), 0.0),
                      at.summarise(fx, "fid_legacy_roundtrip") if len(fx) else (float("nan"), 0.0)]))
    print(at.plain_table(rows, ["SW2 flow", "SW2 floor", "latent SW2", "FID flow", "FID logged-rt"],
                         title=title, note=note, first_column=by, label_width=20))

ladder(LADDER, "n_obs", [5000, 10000, 15000, 20000],
       "Confounded dSprites: observation count (M = 2000, init observed, DR)",
       "FID columns use only the rows with the fixed 2000-image reference")

Confounded dSprites: observation count (M = 2000, init observed, DR)

n_obs               SW2 flow ↓           SW2 floor ↓          latent SW2 ↓         FID flow ↓           FID logged-rt ↓
-----------------------------------------------------------------------------------------------------------------------
5000  (3 seeds)     0.00951 ± 0.00043    0.00999 ± 0.00033    0.04422 ± 0.00249    31.634 ± 2.372       31.842 ± 2.272
10000  (3 seeds)    0.00871 ± 0.00030    0.00996 ± 0.00075    0.04102 ± 0.00126    34.819 ± 1.571       34.844 ± 1.739
15000  (3 seeds)    0.00859 ± 0.00014    0.00949 ± 0.00018    0.03922 ± 0.00143    35.348 ± 0.719       35.733 ± 0.967
20000  (3 seeds)    0.00841 ± 0.00008    0.00970 ± 0.00014    0.03906 ± 0.00047    36.488 ± 1.648       36.448 ± 1.995

FID columns use only the rows with the fixed 2000-image reference


## 5. Initialisation

`observed` is the logged outcome marginal $P_Y$ — this design has a continuous action and
therefore no arms, so the logged sample *is* $P_Y$. The two Gaussians are drawn in the
standardized latent the flow operates in, so $\mathcal N(0, I)$ already matches the
per-coordinate moments and is not a strawman; `moment` additionally matches the full empirical
covariance of the flow's own target support.

**Read this as a convergence statement, not a general one.** All three are scored at the same
fixed budget of 2000 iterations. The same ablation on the single-sprite design, scored only at
the final iteration, produced a null that was later corrected: iteration-resolved, the inits
separate sharply under a *budget* and converge to a common value by ~500 steps. What this table
supports is "the flow reaches the same law from any of the three initialisations under this
budget", not "initialisation does not matter".

In [6]:
INIT = ALL[(ALL.n_obs == 20000) & (ALL.n_particles == 2000) & (ALL.estimator == "dr")]
rows = []
for v in ("observed", "normal", "moment"):
    g = INIT[INIT.init == v]
    if not len(g): continue
    fx = g[g.n_fid_ref == 2000]
    rows.append((f"{v}  ({g.data_seed.nunique()} seeds)",
                 [at.summarise(g, "sw2_flow_mean"),
                  at.summarise(g, "latent_sw2_flow_mean"),
                  at.summarise(g, "mean_transport"),
                  at.summarise(fx, "fid_legacy_flow") if len(fx) else (float("nan"), 0.0)]))
print(at.plain_table(rows, ["SW2 flow", "latent SW2", "transport", "FID"],
                     title="Initialisation, n = 20000, M = 2000, 2000 flow iterations",
                     note="the Gaussian starts travel further to arrive at the same place",
                     first_column="init", label_width=20))

Initialisation, n = 20000, M = 2000, 2000 flow iterations

init                SW2 flow ↓           latent SW2 ↓         transport ↓          FID ↓
----------------------------------------------------------------------------------------
observed  (3 seeds) 0.00841 ± 0.00008    0.03906 ± 0.00047    2.799 ± 0.044        36.488 ± 1.648
normal  (3 seeds)   0.00840 ± 0.00004    0.03904 ± 0.00040    6.207 ± 0.035        36.310 ± 1.661
moment  (3 seeds)   0.00840 ± 0.00006    0.03904 ± 0.00045    5.121 ± 0.075        36.273 ± 1.617

the Gaussian starts travel further to arrive at the same place


## 6. Particle count

The sweep that decides how the headline may be phrased. The flow sits *below* its count-matched
floor at every observation count, and by this project's rule a below-floor value is
over-regularisation rather than accuracy. The test: if it is over-regularisation the gap should
**deepen** as $M$ shrinks; if it were sampling noise the excess would stay flat.

It deepens, monotonically. So the sub-floor reading is a finite-$M$ artefact that shrinks as $M$
grows, and the defensible claim is that the flow **reaches** the count-matched floor — not that it
beats an i.i.d. draw of the same size.

FID is not count-matched across these rungs (the flow row has $M$ images against a reference fixed
at 2000, so below $M = 2000$ it is scored whole; `n_fid_row` records it), which is why this table
leads with SW2. SW2 *is* matched at every rung — `n_floor_matched` tracks $M$.

In [7]:
SWEEP = ALL[(ALL.n_obs == 20000) & (ALL.init == "observed") & (ALL.estimator == "dr")]
rows = []
for m in (250, 500, 1000, 2000):
    g = SWEEP[SWEEP.n_particles == m]
    if not len(g): continue
    rows.append((f"M = {m}  ({g.data_seed.nunique()} seeds)",
                 [at.summarise(g, "sw2_flow_mean"),
                  at.summarise(g, "sw2_floor_noisy_matched_mean"),
                  at.summarise(g, "sw2_excess"), at.summarise(g, "sw2_ratio"),
                  at.summarise(g, "latent_sw2_flow_mean")]))
print(at.plain_table(rows, ["SW2 flow", "SW2 matched floor", "excess", "ratio", "latent SW2"],
                     title="Particle count, n = 20000",
                     note="excess over the COUNT-MATCHED floor, never the raw column. The ratio "
                          "rises monotonically with M, so the sub-floor gap is a finite-M "
                          "artefact: the flow reaches the floor, it does not beat it.",
                     first_column="particles", label_width=22))

Particle count, n = 20000

particles             SW2 flow ↓           SW2 matched floor ↓  excess ↓             ratio ↓              latent SW2 ↓
----------------------------------------------------------------------------------------------------------------------
M = 250  (3 seeds)    0.02242 ± 0.00039    0.03020 ± 0.00049    -0.00778 ± 0.00074   0.74253 ± 0.02101    0.10860 ± 0.00216
M = 500  (3 seeds)    0.01602 ± 0.00035    0.02019 ± 0.00059    -0.00418 ± 0.00058   0.79352 ± 0.02426    0.07776 ± 0.00035
M = 1000  (3 seeds)   0.01141 ± 0.00032    0.01398 ± 0.00079    -0.00257 ± 0.00095   0.81846 ± 0.05690    0.05443 ± 0.00206
M = 2000  (3 seeds)   0.00841 ± 0.00008    0.00970 ± 0.00014    -0.00129 ± 0.00022   0.86722 ± 0.02054    0.03906 ± 0.00047

excess over the COUNT-MATCHED floor, never the raw column. The ratio rises monotonically with M, so the sub-floor gap is a finite-M artefact: the flow reaches the floor, it does not beat it.

## 7. Estimator diagnostics

The headline already carries the three estimators' metrics. What is left here is the diagnostic
side: the total mass each weight vector actually produced before the unit-mass projection, and its
Kish effective size. These are properties of the *estimate*, not of the sampler, and they are the
quantities that would reveal a failing estimator even where the pixel metric cannot.

In [8]:
rows = []
for label, e in NAMES:
    g = PAIRED[PAIRED.estimator == e]
    rows.append((f"{label}  ({g.data_seed.nunique()} seeds)",
                 [at.summarise(g, "sum_v"), at.summarise(g, "sum_abs_v"),
                  at.summarise(g, "v_ess"), at.summarise(g, "mean_transport")]))
print(at.plain_table(rows, ["sum v", "sum |v|", "v_ESS (Kish)", "transport"],
                     title="Target-embedding diagnostics, n = 20000, M = 2000",
                     note="sum v is 1 by construction after the projection; sum |v| above 1 is "
                          "negative mass, and it is tiny here -- the projection is a no-op on "
                          "this design",
                     first_column="estimator", label_width=24))

Target-embedding diagnostics, n = 20000, M = 2000

estimator               sum v ↓              sum |v| ↓            v_ESS (Kish) ↓       transport ↓
--------------------------------------------------------------------------------------------------
DR-IMMD  (3 seeds)      1.000 ± 0.000        1.001 ± 0.000        12187.487 ± 215.645  2.799 ± 0.044
Plug-in IMMD  (3 seeds) 1.000 ± 0.000        1.001 ± 0.000        12234.970 ± 218.354  2.793 ± 0.048
IPW-IMMD  (3 seeds)     1.000 ± 0.000        1.000 ± 0.000        13185.647 ± 134.020  2.801 ± 0.066

sum v is 1 by construction after the projection; sum |v| above 1 is negative mass, and it is tiny here -- the projection is a no-op on this design


## 8. Coverage and provenance

Two caveats that belong beside any table ported from here.

**Hardware.** Pixel SW2 varies by up to **5%** across GPU types — measured 2026-10-06: the same
script on the same GPU twice gives 0.04%, across A100 / L40S / RTX A4500 / Quadro P5000 it reaches
4.9%. The channel is the autoencoder, the only nondeterministic component (`ae_train_mse` moves
−12.9% to +8.2% across card types, while `sum_abs_v`, which never touches it, is identical to
0.000%). Comparisons *within* an experiment run on one card type are safe; cross-experiment
comparisons carry this.

**FID convention.** Only rows with `n_fid_ref == 2000` are comparable; the rest predate the fix.

In [9]:
cov = ALL.groupby(["init", "n_obs", "n_particles"]).agg(
    seeds=("data_seed", "nunique"), gpus=("gpu_name", lambda s: ", ".join(sorted(set(s)))),
    fid_fixed=("n_fid_ref", lambda s: int((s == 2000).sum())), rows=("data_seed", "size"))
print(cov.to_string())
print(f"\nestimated-ratio ESS / n : {(ALL.est_w_ess / ALL.n_obs).min():.2f} .. "
      f"{(ALL.est_w_ess / ALL.n_obs).max():.2f}   (no clipping: weight_clip = "
      f"{sorted(ALL.weight_clip.unique())})")
print(f"target mass sum_v = {sorted(ALL.sum_v.round(6).unique())}   "
      f"sum|v| in [{ALL.sum_abs_v.min():.4f}, {ALL.sum_abs_v.max():.4f}]")

                            seeds                                                   gpus  fid_fixed  rows
init     n_obs n_particles                                                                               
moment   20000 2000             3                                  NVIDIA A100-SXM4-40GB          3     3
normal   20000 2000             3                                  NVIDIA A100-SXM4-40GB          3     3
observed 5000  2000             3  NVIDIA A100-SXM4-40GB, NVIDIA RTX A4500, Quadro P5000          3     6
         10000 2000             3  NVIDIA A100-SXM4-40GB, NVIDIA RTX A4500, Quadro P5000          3     6
         15000 2000             3                NVIDIA A100-SXM4-40GB, NVIDIA RTX A4500          3     6
         20000 250              3                                            NVIDIA L40S          3     3
               500              3                                            NVIDIA L40S          3     3
               1000             3             